<a href="https://colab.research.google.com/github/jpdeleon/quicklook/blob/main/notebook/precession_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Precession search: transit-shape drift across TESS sectors

Nodal precession slowly changes the impact parameter `b` of a transiting planet (or the inclination `i` of an eclipsing binary in a triple), so the transit or eclipse shape drifts between sectors taken years apart. `precession_search=True` fits the shape in each stitched sector and tests for a secular trend. It needs the optional `batman` dependency, installed below with the `[precession]` extra.

Method, flags and outputs: [docs/precession-search.md](https://github.com/jpdeleon/quicklook/blob/main/docs/precession-search.md).

## Setup

On Google Colab, the next cell installs `quicklook` from GitHub. It takes a minute or two the first time, and no runtime restart is needed. Elsewhere it only prints the install command, so it never replaces a local install.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !python -m pip install -q "quicklook-package[precession] @ git+https://github.com/jpdeleon/quicklook.git"
else:
    print('Not on Colab. If needed, install with: pip install "quicklook-package[precession] @ git+https://github.com/jpdeleon/quicklook.git"')

In [ ]:
from importlib.metadata import version

print("quicklook-package", version("quicklook-package"))

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import Image, display

from quicklook import TessQuickLook

OUTDIR = Path("outputs")  # figures and *_tls.h5 files are written here
OUTDIR.mkdir(exist_ok=True)

## Example 1: a planet, TOI-1355 b

Watanabe et al. (2026, arXiv:2608.17387) measured `db/dt = 0.0163 (+0.0008/−0.0008) /yr` for TOI-1355 b. Stitching its five 2-min sectors (57, 58, 77, 78, 84; 2 yr) and restricting TLS to 2.1–2.25 d should give `db/dt ≈ 0.015 /yr` at > 10σ, flagged as a precession candidate, with `T14` falling and `Rp/Rs` flat.

In [ ]:
toi1355 = TessQuickLook(
    "TOI-1355",
    sector="all",
    exptime=120,
    Porb_limits=(2.1, 2.25),
    precession_search=True,
    savefig=True,
    outdir=str(OUTDIR),
    overwrite=True,
)
fig = toi1355.plot_tql()

In [ ]:
p = toi1355.precession_result
print(f"db/dt = {p.db_dt:.4f} ± {p.db_dt_err:.4f} /yr ({p.significance:.1f} sigma, "
      f"dBIC = {p.delta_bic:.1f}) over {p.baseline_yr:.2f} yr")
print("candidate:", p.is_candidate, "| flags:", ", ".join(p.flags) or "none")
for f in p.sector_fits:
    print(f"  S{f.sector}: b = {f.b:.3f} ± {f.b_err:.3f}, T14 = {f.t14 * 24:.2f} h, Rp/Rs = {f.rp_rs:.4f}")
display(Image(filename=next(OUTDIR.glob("TOI1355_*_precession.png"))))

## Example 2: an eclipsing binary in a triple, V907 Sco

Zasche et al. (2023, arXiv:2302.04613) found the inner binary of V907 Sco switching back into eclipses as its inclination grows. V907 Sco only has full-frame-image light curves, at a different cadence in each sector (S13: 30 min, S39: 10 min, S66: 200 s). `TessQuickLook` stitches one cadence only, so here the light curves come straight from `lightkurve` and go to the eclipsing-binary search, `search_eb_precession`. Expect `i` to grow by ~1.8°/yr, and the primary eclipse to deepen from ~2 ppt (S13) to ~150 ppt (S66).

In [ ]:
import lightkurve as lk
import numpy as np

from quicklook.plot_precession import plot_eb_precession
from quicklook.precession import sector_span_from_lc
from quicklook.precession_eb import search_eb_precession

PERIOD = 3.7762049  # d (Zasche et al. 2023, Table 1)
T0 = 2459362.3272 - 2457000  # BTJD of a primary eclipse
DURATION = 5.3 / 24  # d

times, fluxes, errs, spans = [], [], [], []
for row in lk.search_lightcurve("V907 Sco", mission="TESS", author="TESS-SPOC"):
    lc = row.download(quality_bitmask="default").remove_nans().normalize()
    mask = lc.create_transit_mask(
        period=[PERIOD, PERIOD],
        transit_time=[T0, T0 + PERIOD / 2],  # primary and secondary eclipses
        duration=[1.6 * DURATION, 1.6 * DURATION],
    )
    cadence = np.median(np.diff(lc.time.value))
    flat = lc.flatten(window_length=max(int(1.5 / cadence) | 1, 7), mask=mask)
    times.append(flat.time.value)
    fluxes.append(flat.flux.value)
    errs.append(flat.flux_err.value)
    spans.append(sector_span_from_lc(lc))
    print(f"S{spans[-1].sector}: {cadence * 1440:.1f} min cadence, {len(flat)} points")

eb = search_eb_precession(
    np.concatenate(times), np.concatenate(fluxes), np.concatenate(errs),
    spans, PERIOD, T0, DURATION,
)
print(f"di/dt = {eb.di_dt:.2f} ± {eb.di_dt_err:.2f} deg/yr ({eb.significance:.0f} sigma), "
      f"candidate: {eb.is_candidate}, P_nodal <= {eb.max_nodal_period_yr:.0f} yr")
for f in eb.sector_fits:
    print(f"  S{f.sector}: i = {f.inc:.2f} deg, primary depth = {f.depth_data * 1e3:.1f} ppt")
fig = plot_eb_precession(eb, "V907 Sco")

The absolute `i` comes out about 1° above the published values (the radius ratio is poorly constrained in partial eclipses), but the rate agrees: 1.84°/yr between S13 and S39 against 1.68 ± 0.13°/yr from the published inclinations. With the outer orbit and masses, the quadrupole formula (Zasche et al. 2023, eqs. 2–3) gives the precession period itself:

In [ ]:
from quicklook.precession_eb import nodal_precession_period

p_nodal = nodal_precession_period(
    p_inner=3.7762049, p_outer=142.01,  # days
    m1a=2.56, m1b=2.74, m_third=1.06,  # solar masses
    mutual_inc_deg=26.2, e_outer=0.467,
)
print(f"P_nodal = {p_nodal / 365.25:.1f} yr (published: 63.5 +3.3/-2.6 yr)")

## Same runs from the command line

```bash
ql run --name TOI-1355 --sector all --exptime 120 --period-limits 2.1 2.25 \
  --precession-search --save
# eclipsing binary with one cadence: give the binary period and primary T0
ql run --name <EB> --sector all --exptime 120 --precession-search --precession-mode eb \
  --custom-ephem <T0> <T0err> <P> <Perr> <T14 [h]> <T14err> --save
```

In the web GUI: Sector `all`, Exposure Time `120`, Period Min/Max `2.1`/`2.25`, and tick **Precession search** (mode **Auto**, **Planet** or **EB**).